# CS781 Phase 2 — Notebook P2_02: fine-tune the Phase-1 models for Legal NER

One notebook, many experiments. Each experiment = one small config dict (Phase-1 model, attention mode,
fine-tuning method, head, context, class balancing). The notebook runs a **queue** of experiments — one per GPU in
parallel (Kaggle has 2×T4) — and for every experiment writes dev scores, an error analysis and a ready
`submission.zip`.

| Step | What happens |
|---|---|
| 1 | config: where the Phase-1 weights are, the experiment list, which queue to run |
| 2 | inputs: P2_01's documents (tokens, offsets, BIO tags, dev split) + the five Phase-1 checkpoints |
| 3 | model classes (copied unchanged from NB3) |
| 4 | **backbone** = a Phase-1 model without its LM head → one vector per token; causal or bidirectional attention |
| 5 | **heads**: linear softmax, CRF (our own forward algorithm + Viterbi), optional BiLSTM in front |
| 6 | **adaptation methods**: full, frozen (linear probe), top-k layers, LoRA, **LoRMA** (+ and π), BitFit, adapters |
| 7 | windows over long documents (+ optional document header), class weights, oversampling |
| 8 | inference: overlapping windows merged per token → decoding (greedy / constrained Viterbi) → character offsets |
| 9 | dev-set tuning (per-label decision biases, document-level label consistency) + error analysis |
| 10 | training loop, experiment runner (dev model → test predictions; optional retrain on all 70 docs) |
| 11 | unit tests (CPU, seconds): CRF vs brute force, adapters = identity at init, windows reproduce the gold |
| 12 | run the queue on all GPUs |
| 13 | results table across all experiments (also from attached earlier runs) |

**Kaggle settings**: Accelerator **GPU T4 x2**, Internet not needed.
**Inputs** (Add Input → Your Work / Datasets):
1. the **P2_01 output** (`p2/data/docs.pkl`, `meta.json`)
2. the **Phase-1 weight datasets** `cs781-p1-a0`, `cs781-p1-a1`, `cs781-p1-b`, `cs781-p1-e`, `cs781-p1-f`
   (each containing `cs781_p1/runs/<run_name>/weights_final.pt` + `run_info.json`) — names editable in step 1
3. *(optional)* earlier versions of **this** notebook: finished experiments are recognised and skipped, and the
   results table includes them.

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
HF token found: True


## 1. Configuration
* `P1_SOURCES`: dataset name + run name of each Phase-1 model (edit if your datasets are named differently).
* `EXPERIMENTS`: every experiment = changes relative to `DEFAULTS`.
* `RUN_QUEUE`: which experiments this commit runs. Run different queues on different accounts in parallel.

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F, pickle, zipfile, threading, queue as _queue, traceback, csv, copy
import pandas as pd
try: display
except NameError: display = print

P2_SMOKE = os.environ.get("CS781_P2_SMOKE") == "1"      # local pipeline test only (tiny models)

# ---- where the Phase-1 models are (Kaggle dataset name, run folder name) ----
P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-a0", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-a1", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-b",  run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="cs781-p1-e",  run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-f",  run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}

DEFAULTS = dict(
    model="E",              # Phase-1 backbone: A0 | A1 | B | E | F
    attn="bidir",           # "bidir" (every token sees the whole window) | "causal" (as in pretraining). B is always bidir.
    init="pretrained",      # "pretrained" | "random" (same architecture, no Phase-1 knowledge = transfer control)
    method="full",          # full | frozen | topk | lora | lorma | bitfit | adapter
    lr=None,                # backbone learning rate (None -> METHOD_LR[method]); the head always uses lr_head
    lora_r=16, lora_alpha=32, lora_targets="all",   # LoRA/LoRMA rank, scale, which linears ("attn" | "all")
    lorma_variant="plus",   # "plus": (I + a/r·BA)·W0   |   "pi": permutation-inflated (BA)·W0
    topk=4,                 # method="topk": number of top blocks trained
    adapter_dim=64,         # method="adapter": bottleneck size
    head="linear",          # "linear" (softmax per token) | "crf"
    lstm_hidden=0,          # >0: BiLSTM(hidden per direction) between backbone and head
    head_dropout=0.1, backbone_dropout=0.1,
    win_len=512,            # tokens per window (incl. header)
    header_len=0,           # >0: the first header_len tokens of the document are prepended to every window
    train_stride=None,      # None -> content length (non-overlapping, random offset every epoch)
    infer_stride=None,      # None -> half the content length (every token seen by 2 windows, central one weighted most)
    epochs=15, batch=8, lr_head=1e-3, weight_decay=0.01, warmup=0.06, grad_clip=1.0,
    class_weight="sqrt_inv",  # None | "sqrt_inv": weight_c = sqrt(count_O / count_c), capped at 10 (softmax head only)
    oversample=0,           # k>0: windows containing a rare label are repeated k extra times per epoch
    tune_bias=True,         # tune per-label decision biases on dev (kept only if dev macro-F1 improves by ≥0.005)
    final="dev_model",      # "dev_model": test predictions from the model selected on dev | "retrain_all": retrain on all 70 docs
    teacher=None,           # knowledge distillation: name of a finished experiment (linear head) used as the teacher
    kd_alpha=0.5, kd_T=2.0, # loss = alpha·CE(gold) + (1-alpha)·T²·KL(teacher_T || student_T)
    seed=0,
)
METHOD_LR = {"full": 5e-5, "frozen": 0.0, "topk": 1e-4, "lora": 5e-4, "lorma": 5e-4, "bitfit": 1e-3, "adapter": 1e-3}

EXPERIMENTS = {
    # ---- Stage 1: transfer screen — full fine-tuning + linear head on every Phase-1 model ----
    "s1_A0_full_causal": dict(model="A0", attn="causal"),
    "s1_A0_full_bidir":  dict(model="A0", attn="bidir"),
    "s1_A1_full_causal": dict(model="A1", attn="causal"),
    "s1_A1_full_bidir":  dict(model="A1", attn="bidir"),
    "s1_E_full":         dict(model="E"),
    "s1_F_full_causal":  dict(model="F", attn="causal"),
    "s1_F_full_bidir":   dict(model="F", attn="bidir"),
    "s1_B_full":         dict(model="B"),
    "s1_A1_randominit":  dict(model="A1", attn="bidir", init="random", lr=3e-4),   # no Phase-1 knowledge
    # ---- Stage 2: design ablations on the Stage-1 winner (E; causal-pretrained models stay causal) ----
    "s2_E_crf":          dict(model="E", head="crf"),
    "s2_E_bilstm":       dict(model="E", lstm_hidden=256),
    "s2_E_header128":    dict(model="E", header_len=128),
    "s2_E_win1024":      dict(model="E", win_len=1024, batch=4),
    "s2_E_noclassw":     dict(model="E", class_weight=None),
    "s2_E_oversample":   dict(model="E", oversample=1),
    "s2_A1_causal_bilstm": dict(model="A1", attn="causal", lstm_hidden=256),
    "s2_E_crf_header128": dict(model="E", head="crf", header_len=128),
    "s2_E_header256":    dict(model="E", header_len=256),
    # ---- Stage 3: adaptation methods on one backbone ----
    "s3_E_frozen":       dict(model="E", method="frozen", epochs=25),
    "s3_E_top4":         dict(model="E", method="topk", topk=4, epochs=20),
    "s3_E_lora16":       dict(model="E", method="lora", epochs=25),
    "s3_E_lorma16":      dict(model="E", method="lorma", epochs=25),
    "s3_E_lorma_pi":     dict(model="E", method="lorma", lorma_variant="pi", lora_targets="attn", epochs=25),
    "s3_E_bitfit":       dict(model="E", method="bitfit", epochs=25),
    "s3_E_adapter64":    dict(model="E", method="adapter", epochs=25),
}
# ---- FINAL submissions: best Stage-2/3 settings, retrained on all 70 documents ----
FINAL_COMMON = dict(header_len=256, oversample=1, final="retrain_all")
EXPERIMENTS.update({
    "f1_A1_causal_bilstm_full": dict(model="A1", attn="causal", lstm_hidden=256, **FINAL_COMMON),
    "f2_E_lorma":               dict(model="E", method="lorma", epochs=25, **FINAL_COMMON),
    "f3_F_causal_adapter":      dict(model="F", attn="causal", method="adapter", lstm_hidden=256, epochs=25, **FINAL_COMMON),
    "f4_B_kd":                  dict(model="B", teacher="s2_A1_causal_bilstm", **FINAL_COMMON),   # student = B's 6-layer encoder
    "f4_B_nokd":                dict(model="B", header_len=256, oversample=1),                    # same student without KD (control)
})
QUEUES = {"stage1": [k for k in EXPERIMENTS if k.startswith("s1_")],
          "stage2": [k for k in EXPERIMENTS if k.startswith("s2_")],
          "stage3": [k for k in EXPERIMENTS if k.startswith("s3_")],
          "final_acc1": ["f4_B_kd", "f1_A1_causal_bilstm_full", "f4_B_nokd"],   # account with the Stage-2 output attached (KD teacher)
          "final_acc2": ["f2_E_lorma", "f3_F_causal_adapter"]}
RUN_QUEUE = QUEUES["final_acc2"]          # <- choose what this commit runs (a list of experiment names)
MIN_HOURS_PER_EXP = 0.6               # do not start another experiment with less time than this left

if P2_SMOKE:   # tiny smoke-test checkpoints from NB3, few epochs (local test of the whole pipeline)
    P1_SOURCES = {k: dict(dataset="", run_name=n) for k, n in {
        "A0": "SMOKE_A0_gpt2_110M_mix50_300M", "A1": "SMOKE_A1_modern_110M_mix50_300M",
        "B": "SMOKE_B_encdec_6x6_mix50_epoch_300M", "E": "SMOKE_E_prefixlm_110M_mix50_epoch_300M",
        "F": "SMOKE_F_localglobal_gqa_alibi_mix50_epoch_300M"}.items()}
    DEFAULTS.update(epochs=2, win_len=128, batch=4)
    for k in EXPERIMENTS:
        if EXPERIMENTS[k].get("win_len"): EXPERIMENTS[k]["win_len"] = 256
        if EXPERIMENTS[k].get("header_len"): EXPERIMENTS[k]["header_len"] = 32
        EXPERIMENTS[k]["epochs"] = 2
    RUN_QUEUE = [x for x in os.environ.get("CS781_P2_QUEUE", ",".join(QUEUES["stage1"])).split(",") if x]

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    assert cfg["header_len"] <= cfg["win_len"] // 2, f"{name}: header_len must be at most half of win_len"
    return cfg

EXPS_DIR = ROOT / "p2" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
try:
    NUM_GPUS = torch.cuda.device_count()
except Exception:
    NUM_GPUS = 0
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['f2_E_lorma', 'f3_F_causal_adapter']


## 2. Inputs
* P2_01 output: every document with token ids, character offsets of each token, BIO tags and the dev split.
* Earlier versions of this notebook (if attached): their small result files are copied in, so finished
  experiments are skipped and appear in the results table.
* Phase-1 checkpoints: located by dataset name, then by run-folder name anywhere under `/kaggle/input`.

In [3]:
DOCS_PATH, META_PATH = find_artifact("p2/data/docs.pkl"), find_artifact("p2/data/meta.json")
assert DOCS_PATH is not None, "attach the P2_01 output (p2/data/docs.pkl)"
with open(DOCS_PATH, "rb") as f: DOCS = pickle.load(f)
META = load_json(META_PATH)
LABELS, BIO = META["labels"], META["bio"]; NT = len(BIO); END_SHIFT = META["end_shift"]
PAD_ID, SEP_ID = 1, 0                     # <|pad|> and <|endoftext|> (separator between header and window)
TRAIN_DOCS = [d for d in DOCS if d["split"] == "train"]
DEV_DOCS   = [d for d in DOCS if d["split"] == "dev"]
TEST_DOCS  = [d for d in DOCS if d["split"] == "test"]
print(f"docs: train {len(TRAIN_DOCS)} | dev {len(DEV_DOCS)} | test {len(TEST_DOCS)} | align mode {META['align_mode']} "
      f"| empty docs: {sum(len(d['ids']) == 0 for d in DOCS)}")

# results of earlier versions (small files only — weights stay where they are and are found by find_artifact)
if IN_KAGGLE:
    n = 0
    for cand in glob.glob(f"/kaggle/input/**/{PROJECT}/p2/exps/*", recursive=True):
        for src in Path(cand).glob("*"):
            if src.suffix in (".json", ".zip", ".txt", ".jsonl"):
                dst = EXPS_DIR / Path(cand).name / src.name
                if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
    print("restored", n, "result files from attached earlier runs")

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = []
    for r in roots:
        cands += glob.glob(f"{r}/**/weights_final.pt", recursive=True)
    cands = sorted(set(cands))
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None

P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND — attach its dataset or fix P1_SOURCES'}")
needed = {exp_cfg(n)["model"] for n in RUN_QUEUE}
missing = [k for k in needed if P1_DIRS[k] is None]
assert not missing, f"Phase-1 weights missing for {missing}"

docs: train 56 | dev 14 | test 35 | align mode punct | empty docs: 1
restored 0 result files from attached earlier runs
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (unchanged from NB3)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

## 4. Backbone: a Phase-1 model as a token encoder
We drop the LM head and return the final (normalised) hidden state of every token. What changes is **who attends
to whom**:

| model | `attn="causal"` | `attn="bidir"` |
|---|---|---|
| A0, A1 | as pretrained: token i sees 0..i | causal mask removed: every token sees the whole window (weights must adapt) |
| E prefix-LM | (ablation only) | the whole window is the "prefix" — exactly the bidirectional mode E was pretrained with |
| F local-global | as pretrained (window 256 looks back) | symmetric: ±256 in local layers, everything in global layers, ALiBi on |distance| |
| B enc-dec | — | the **encoder** only (bidirectional by design); the decoder is discarded |

NER needs right context ("Ram Kumar, *the appellant*"), so bidirectional attention should help — but it is a change
from pretraining for A0/A1/F. Stage 1 measures both.

In [5]:
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec":
            lm.dec = nn.ModuleList()              # decoder not used for NER (drops ~40M parameters)
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)

    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks

    def forward(self, idx, valid):
        lm, (B, T) = self.lm, idx.shape
        assert T <= self.ctx_limit, f"window {T} longer than the model's position range {self.ctx_limit}"
        dev = idx.device
        if self.arch == "encdec":
            return lm.encode(idx, valid)
        mask = valid[:, None, None, :] if self.bidir else None        # bool: True = may attend (keys)
        pos = torch.arange(T, device=dev)[None]
        if self.arch == "gpt2":
            x = lm.drop(lm.wte(idx) + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            return lm.ln_f(x)
        if self.arch in ("modern", "prefixlm"):
            x = lm.wte(idx)
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            return lm.norm_f(x)
        if self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()                     # (1, T, T) query - key
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(T, dtype=torch.bool, device=dev)[None]      # padded queries keep one key
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            x = lm.wte(idx)
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            return lm.norm_f(x)
        raise ValueError(self.arch)

## 5. Heads: linear softmax and CRF

**Linear**: `emission = W·h + b` → softmax over the 25 BIO tags, independently per token.

**CRF** (conditional random field): adds a learned score for every tag→tag transition, so the model scores whole tag
*sequences*: `score(y) = Σ_t emission_t[y_t] + Σ_t trans[y_{t-1}, y_t] + start[y_0] + end[y_T]`, trained with
`−log p(y_gold) = log Z − score(y_gold)`, where `log Z` (all 25^T sequences) is computed with the **forward algorithm**
(dynamic programming, O(T·25²)). Decoding = **Viterbi** (best sequence). Illegal BIO moves (`O → I-X`, `B-X → I-Y`)
get a fixed −10⁴, so the CRF can never output a broken entity. The same constrained Viterbi (with zero learned
transitions) is also available as a decoder for the linear head.

**BiLSTM** (optional, `lstm_hidden>0`): reads the backbone's vectors left→right and right→left — a cheap way to give
a *causal* backbone right-hand context.

In [6]:
def bio_constraints():
    allowed = np.ones((NT, NT), dtype=bool)
    for j, t in enumerate(BIO):
        if t.startswith("I-"):
            for i, s in enumerate(BIO): allowed[i, j] = s in ("B-" + t[2:], t)
    start_ok = np.array([not t.startswith("I-") for t in BIO])
    return allowed, start_ok
ALLOWED, START_OK = bio_constraints()
NEG = -1e4

class CRF(nn.Module):
    def __init__(self, n, allowed, start_ok):
        super().__init__()
        self.trans, self.start, self.end = nn.Parameter(torch.zeros(n, n)), nn.Parameter(torch.zeros(n)), nn.Parameter(torch.zeros(n))
        self.register_buffer("cmask", torch.where(torch.tensor(allowed), 0.0, NEG), persistent=False)
        self.register_buffer("smask", torch.where(torch.tensor(start_ok), 0.0, NEG), persistent=False)
    def full(self):
        return self.trans + self.cmask, self.start + self.smask, self.end
    def nll(self, em, tags, mask):
        """em (B,T,n) float32, tags (B,T) long (anything where mask is False), mask (B,T) bool, True-prefix per row"""
        tr, st, en = self.full()
        tags = tags.clamp(min=0)
        score = st[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        alpha = st[None] + em[:, 0]
        for t in range(1, em.size(1)):
            m = mask[:, t]
            score = score + (tr[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)) * m
            nxt = torch.logsumexp(alpha[:, :, None] + tr[None] + em[:, t, None, :], dim=1)
            alpha = torch.where(m[:, None], nxt, alpha)
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        logZ = torch.logsumexp(alpha + en[None], 1)
        return (logZ - score - en[last]).sum() / mask.sum().clamp(min=1)

class TokenHead(nn.Module):
    def __init__(self, d, n, kind="linear", lstm_hidden=0, dropout=0.1):
        super().__init__()
        self.kind, self.drop = kind, nn.Dropout(dropout)
        self.lstm = nn.LSTM(d, lstm_hidden, batch_first=True, bidirectional=True) if lstm_hidden else None
        self.proj = nn.Linear(2 * lstm_hidden if lstm_hidden else d, n)
        self.crf = CRF(n, ALLOWED, START_OK) if kind == "crf" else None
    def forward(self, h, valid):
        with torch.autocast(device_type=h.device.type, enabled=False):          # head in fp32 (small)
            h = self.drop(h.float())
            if self.lstm is not None:
                lengths = valid.sum(1).clamp(min=1).cpu()
                packed = nn.utils.rnn.pack_padded_sequence(h, lengths, batch_first=True, enforce_sorted=False)
                out, _ = self.lstm(packed)
                h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.size(1))
            return self.proj(h)

class NERModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__(); self.backbone, self.head = backbone, head
    def forward(self, idx, valid):
        return self.head(self.backbone(idx, valid), valid)

## 6. Adaptation methods (all written here, no libraries)

Each method decides which numbers change during fine-tuning. `W0` = a frozen pretrained weight matrix.

| method | layer output | trainable |
|---|---|---|
| `full` | `W x` | everything |
| `frozen` (linear probe) | `W0 x` | head only — measures how much NER information Phase-1 already put in the vectors |
| `topk` (partial / layer-wise) | top k blocks trainable | ~k/12 of the backbone |
| `lora` (Hu et al. 2021) | `W0 x + (α/r)·B A x`,  A∈ℝ^{r×d_in}, B∈ℝ^{d_out×r}, B=0 at start | ~1–2 % |
| `lorma` **+** (Bihany et al. 2025) | `(I + (α/r)·B A)·W0 x` — a low-rank matrix **multiplies** the frozen output; B=0 at start | same as LoRA |
| `lorma` **π** | `I_π(B A)·W0 x`; `I_π` cyclically shifts row i by i ("rank inflation" without the +I); init makes `I_π(BA)=I` | same |
| `bitfit` (Ben Zaken et al. 2022) | `W0 x + b` | biases (+ norm gains) only; our bias-free models get new zero biases |
| `adapter` (Houlsby et al. 2019) | `y + U·GELU(D·y)` after each attention/MLP output projection, U=0 at start | ~1 % |

**Knowledge distillation** (`teacher=<experiment>`, Hinton et al. 2015): a finished, larger fine-tuned model gives a
probability distribution over the 25 tags for every token. The student is trained on
`α·CE(gold) + (1−α)·T²·KL(softmax(teacher/T) ‖ softmax(student/T))`. The temperature T>1 softens the teacher's
distribution so the student also learns *which wrong tags are plausible* (e.g. RESP vs APP), information that the
one-hot gold label does not contain.

Every method starts **exactly** at the pretrained function (checked by the unit tests): the extra matrices are zero
(or the identity for LoRMA-π) at initialisation, so fine-tuning begins from what Phase-1 learned.
LoRA *adds* a low-rank correction to the weights; LoRMA *rotates/scales the output space* of the frozen layer.
Computing LoRMA⁺ as `y + (α/r)·B(A y)` with `y = W0 x` never forms a d×d matrix.

In [7]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device)
            _PERM_IDX[key] = (i[None, :] - i[:, None]) % d          # I_pi(M)[i, j] = M[i, (j - i) mod d]
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:                                                      # permutation-based rank inflation
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A                                        # (d, d), rank <= r
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    """(parent, attribute, linear, is_output_projection) for the attention/MLP linears of the blocks in use"""
    out = []
    for parent in backbone.blocks().modules():
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        final_norm = backbone.lm.enc_norm if backbone.arch == "encdec" else (backbone.lm.ln_f if backbone.arch == "gpt2" else backbone.lm.norm_f)
        for p in final_norm.parameters(): p.requires_grad = True
        return
    which = cfg["lora_targets"] if m in ("lora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m == "lora":    setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"]))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":                                          # norm gains/biases are the other "bias-like" parameters
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

### Loading a Phase-1 checkpoint and assembling the NER model

In [8]:
_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {})
            assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            sd = {k: v.float() if v.is_floating_point() else v for k, v in sd.items()}
            _P1_CACHE[key] = (pcfg, sd)
        return _P1_CACHE[key]

def build_ner_model(cfg, device):
    pcfg, sd = load_phase1(cfg["model"])
    vocab = sd["wte.weight"].shape[0]
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=vocab, **{k: v for k, v in pcfg.items() if k != "vocab"})
    if cfg["init"] == "pretrained":
        lm.load_state_dict(sd)
    backbone = Backbone(lm, pcfg["arch"], cfg["attn"])
    set_backbone_dropout(backbone, cfg["backbone_dropout"])
    apply_method(backbone, cfg)
    head = TokenHead(backbone.d, NT, cfg["head"], cfg["lstm_hidden"], cfg["head_dropout"])
    model = NERModel(backbone, head).to(device)
    return model, pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

## 7. Windows, class weights, oversampling

Documents average ~6.7k tokens; the models read windows of `win_len` tokens.
* **Training**: non-overlapping windows with a random start offset each epoch (different cuts every epoch).
  A window that starts in the middle of an entity gets `B-` on its first token (so the CRF never sees an illegal start).
* **Inference**: windows overlap by half; each token's scores are a weighted average over the windows that contain
  it, weighted by distance to the window edge — the window where the token is most central counts most.
* **Header** (`header_len>0`): the first tokens of the document (cause title: parties, court, case number) +
  a separator are prepended to *every* window, so later mentions of the parties can be resolved. No loss on them.

In [9]:
def windows_for(doc, cfg, train, rng=None):
    ids = doc["ids"]; N = len(ids); L, H = cfg["win_len"], cfg["header_len"]
    if N == 0: return []
    use_header = H > 0 and N > L
    Lc = L - (H + 1 if use_header else 0)
    if N <= Lc: starts = [0]
    else:
        stride = (cfg["train_stride"] or Lc) if train else (cfg["infer_stride"] or max(1, Lc // 2))
        off = int(rng.integers(0, stride)) if (train and rng is not None) else 0
        starts = sorted({0, N - Lc, *[s for s in range(off, N - Lc, stride)]})
    out = []
    head = np.concatenate([ids[:H], [SEP_ID]]).astype(np.int64) if use_header else np.zeros(0, np.int64)
    for s in starts:
        w = ids[s:s + Lc].astype(np.int64)
        ex = {"doc": doc["id"], "start": s, "len": len(w), "off": len(head), "ids": np.concatenate([head, w])}
        if doc.get("tags") is not None:
            t = doc["tags"][s:s + Lc].astype(np.int64).copy()
            if train and len(t) and t[0] > 0 and BIO[t[0]].startswith("I-"): t[0] -= 1        # I-X -> B-X
            ex["tags"] = t
        out.append(ex)
    return out

def collate(exs, device):
    B, T, Lc = len(exs), max(len(e["ids"]) for e in exs), max(e["len"] for e in exs)
    idx = np.full((B, T), PAD_ID, np.int64); valid = np.zeros((B, T), bool)
    tags = np.full((B, Lc), -100, np.int64); cmask = np.zeros((B, Lc), bool)
    offs = np.array([e["off"] for e in exs], np.int64)
    for b, e in enumerate(exs):
        idx[b, :len(e["ids"])] = e["ids"]; valid[b, :len(e["ids"])] = True
        cmask[b, :e["len"]] = True
        if "tags" in e: tags[b, :e["len"]] = e["tags"]
    t = lambda a: torch.from_numpy(a).to(device, non_blocking=True)
    return t(idx), t(valid), t(offs), t(tags), t(cmask)

def content(em, offs, Lc):
    """(B, T, n) emissions -> (B, Lc, n) for the window part (drops header + separator)"""
    gi = (offs[:, None] + torch.arange(Lc, device=em.device)[None]).clamp(max=em.size(1) - 1)
    return em.gather(1, gi[:, :, None].expand(-1, -1, em.size(-1)))

TAG_COUNTS = np.bincount(np.concatenate([d["tags"] for d in TRAIN_DOCS + DEV_DOCS if len(d["ids"])]), minlength=NT).astype(float)
RARE_LABELS = sorted(LABELS, key=lambda l: TAG_COUNTS[BIO.index("B-" + l)])[:6]
RARE_TAGS = {BIO.index(p + l) for l in RARE_LABELS for p in ("B-", "I-")}
def class_weights(kind):
    if kind is None: return None
    w = np.sqrt(TAG_COUNTS[0] / np.maximum(TAG_COUNTS, 1)); w = np.minimum(w, 10.0); w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32)
print("rare labels (oversampled when oversample>0):", RARE_LABELS)
print("class weights (sqrt_inv):", dict(zip(BIO, class_weights("sqrt_inv").numpy().round(2))))

rare labels (oversampled when oversample>0): ['R.COUNSEL', 'WIT', 'A.COUNSEL', 'JUDGE', 'RESP', 'APP']
class weights (sqrt_inv): {'O': np.float32(1.0), 'B-APP': np.float32(10.0), 'I-APP': np.float32(10.0), 'B-RESP': np.float32(10.0), 'I-RESP': np.float32(10.0), 'B-A.COUNSEL': np.float32(10.0), 'I-A.COUNSEL': np.float32(10.0), 'B-R.COUNSEL': np.float32(10.0), 'I-R.COUNSEL': np.float32(10.0), 'B-JUDGE': np.float32(10.0), 'I-JUDGE': np.float32(10.0), 'B-WIT': np.float32(10.0), 'I-WIT': np.float32(10.0), 'B-AUTH': np.float32(10.0), 'I-AUTH': np.float32(10.0), 'B-COURT': np.float32(10.0), 'I-COURT': np.float32(10.0), 'B-STAT': np.float32(10.0), 'I-STAT': np.float32(10.0), 'B-PREC': np.float32(10.0), 'I-PREC': np.float32(5.97), 'B-DATE': np.float32(10.0), 'I-DATE': np.float32(8.26), 'B-CASENO': np.float32(10.0), 'I-CASENO': np.float32(8.54)}


## 8. Inference, decoding, offsets

Per document: merged per-token scores (log-probabilities for the linear head, raw emissions for the CRF) →
* **greedy**: argmax per token, then BIO → entities (a `B-X`, or an `I-X` not continuing an `X`, starts an entity);
* **viterbi**: best *legal* BIO sequence (CRF transitions, or zero transitions + the BIO constraints for the linear head).

Optional **decision biases** shift the scores before decoding (`−β_O` on `O`, `+β_label` on that label's B/I tags),
trading precision for recall per label — macro-F1 rewards recall on the rare labels.
Entity tokens → characters: core of the first token … end of the last token in normalised text → original offsets.

In [10]:
def bio_to_token_spans(tags):
    out, cur = [], None
    for i, t in enumerate(tags):
        t = int(t)
        if t == 0:
            if cur: out.append(tuple(cur)); cur = None
            continue
        typ, begin = (t - 1) // 2, (t - 1) % 2 == 0
        if cur and not begin and cur[2] == typ: cur[1] = i
        else:
            if cur: out.append(tuple(cur))
            cur = [i, i, typ]
    if cur: out.append(tuple(cur))
    return out

def doc_spans_from_tags(doc, tags):
    core, end, idx, n = doc["tok_core"], doc["tok_end"], doc["idx"], len(doc["norm"])
    out = set()
    for a, b, typ in bio_to_token_spans(tags):
        cs, ce = int(core[a]), min(int(end[b]), n)
        if ce <= cs: continue
        out.add((int(idx[cs]), int(idx[ce - 1]) + 1, LABELS[typ]))
    return sorted(out)

def viterbi_batch(ems, trans, start, end):
    """ems: list of (N_i, n) float arrays -> list of best tag sequences (vectorised over documents)"""
    D, n = len(ems), trans.shape[0]
    lens = np.array([len(e) for e in ems]); Nmax = max(lens.max(initial=0), 1)
    E = np.zeros((D, Nmax, n), np.float32); M = np.zeros((D, Nmax), bool)
    for i, e in enumerate(ems): E[i, :len(e)] = e; M[i, :len(e)] = True
    score = start[None] + E[:, 0]; bp = np.zeros((D, Nmax, n), np.int16); ar = np.arange(n)
    for t in range(1, Nmax):
        cand = score[:, :, None] + trans[None]
        best = cand.argmax(1); val = np.take_along_axis(cand, best[:, None, :], 1)[:, 0] + E[:, t]
        m = M[:, t][:, None]
        score = np.where(m, val, score); bp[:, t] = np.where(m, best, ar[None])
    score = score + end[None]
    out = []
    for i in range(D):
        if lens[i] == 0: out.append(np.zeros(0, np.int64)); continue
        path = np.zeros(lens[i], np.int64); path[-1] = score[i].argmax()
        for t in range(lens[i] - 1, 0, -1): path[t - 1] = bp[i, t, path[t]]
        out.append(path)
    return out

def bias_vector(bias):
    v = np.zeros(NT, np.float32)
    if not bias: return v
    v[0] -= bias.get("O", 0.0)
    for j, t in enumerate(BIO[1:], 1): v[j] += bias.get(t[2:], 0.0)
    return v

def decode(ems, docs, how, crf_params=None, bias=None, consistency=False):
    bv = bias_vector(bias)
    adj = [e + bv[None] for e in (ems[d["id"]] for d in docs)]
    if how == "greedy":
        tagseqs = [a.argmax(1) if len(a) else np.zeros(0, np.int64) for a in adj]
    else:
        if crf_params is None:
            trans, start, end = np.where(ALLOWED, 0.0, NEG).astype(np.float32), np.where(START_OK, 0.0, NEG).astype(np.float32), np.zeros(NT, np.float32)
        else:
            trans, start, end = crf_params
        tagseqs = viterbi_batch(adj, trans, start, end)
    preds = {}
    for d, tg in zip(docs, tagseqs):
        sp = doc_spans_from_tags(d, tg) if len(tg) else []
        preds[d["id"]] = relabel_consistent(d, sp) if consistency else sp
    return preds

def relabel_consistent(doc, spans):
    """the same string never has two labels within one gold document -> majority label per string"""
    votes = defaultdict(Counter)
    for s, e, l in spans: votes[doc["text"][s:e]][l] += 1
    return sorted({(s, e, votes[doc["text"][s:e]].most_common(1)[0][0]) for s, e, l in spans})

def strict_scores(truth, pred):
    """same arithmetic as the official evaluate_phase2.py"""
    stats = {l: [0, 0, 0] for l in LABELS}
    for did in truth:
        g = set(map(tuple, truth[did])); p = set(map(tuple, pred.get(did, [])))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = {"P": pr, "R": rc, "F1": 2 * pr * rc / (pr + rc) if pr + rc else 0.0, "tp": tp, "fp": fp, "fn": fn}
    return {"macro_f1": float(np.mean([v["F1"] for v in per.values()])), "macro_p": float(np.mean([v["P"] for v in per.values()])),
            "macro_r": float(np.mean([v["R"] for v in per.values()])), "per_label": per}

TRUTH_DEV = {d["id"]: [tuple(x) for x in d["spans"]] for d in DEV_DOCS}

@torch.no_grad()
def doc_emissions(model, docs, cfg, device, batch=16):
    model.eval()
    use_amp = device.type == "cuda"
    acc = {d["id"]: np.zeros((len(d["ids"]), NT), np.float32) for d in docs}
    wsum = {d["id"]: np.zeros(len(d["ids"]), np.float32) for d in docs}
    exs = [ex for d in docs for ex in windows_for(d, cfg, train=False)]
    for i in range(0, len(exs), batch):
        chunk = exs[i:i + batch]
        idx, valid, offs, _, _ = collate(chunk, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            em = model(idx, valid)
        em = content(em.float(), offs, max(e["len"] for e in chunk))
        if model.head.crf is None: em = F.log_softmax(em, -1)
        em = em.cpu().numpy()
        for b, e in enumerate(chunk):
            n = e["len"]; j = np.arange(n); w = np.minimum(j + 1, n - j).astype(np.float32)
            acc[e["doc"]][e["start"]:e["start"] + n] += em[b, :n] * w[:, None]
            wsum[e["doc"]][e["start"]:e["start"] + n] += w
    return {k: acc[k] / np.maximum(wsum[k], 1e-6)[:, None] for k in acc}

def crf_numpy(model):
    if model.head.crf is None: return None
    tr, st, en = model.head.crf.full()
    return tuple(x.detach().float().cpu().numpy() for x in (tr, st, en))

## 9. Dev tuning and error analysis

On the 14 dev documents we choose, in order: decoder (greedy vs Viterbi) → per-label decision biases (coordinate
search; kept only if it gains ≥ 0.005) → document-level label consistency (on/off). All choices are stored and
applied unchanged to the test set.

**Error analysis** splits every miss/false alarm into the categories the task asks about:
`boundary` (right label, overlapping but not identical span), `label` (exact span, wrong label), `boundary+label`,
`missed` (no overlapping prediction) and `spurious` (prediction overlapping no gold entity — e.g. boilerplate).

In [11]:
def tune_on_dev(ems, cfg, crf_params, log):
    res = {}
    hows = ["viterbi"] if crf_params is not None else ["greedy", "viterbi"]
    for how in hows:
        res[f"raw_{how}"] = strict_scores(TRUTH_DEV, decode(ems, DEV_DOCS, how, crf_params))["macro_f1"]
    how = max(hows, key=lambda h: res[f"raw_{h}"])
    base = res[f"raw_{how}"]; bias = {}
    if cfg["tune_bias"]:
        cur, best = {}, base
        f = lambda b: strict_scores(TRUTH_DEV, decode(ems, DEV_DOCS, how, crf_params, b))["macro_f1"]
        for v in (-1.5, -1.0, -0.5, 0.5, 1.0, 1.5):
            s = f({**cur, "O": v})
            if s > best: best, cur = s, {**cur, "O": v}
        for l in LABELS:
            for v in (-1.0, -0.5, 0.5, 1.0):
                s = f({**cur, l: v})
                if s > best: best, cur = s, {**cur, l: v}
        res["tuned"] = best
        if best >= base + 0.005: bias = cur
    s0 = strict_scores(TRUTH_DEV, decode(ems, DEV_DOCS, how, crf_params, bias))["macro_f1"]
    s1 = strict_scores(TRUTH_DEV, decode(ems, DEV_DOCS, how, crf_params, bias, consistency=True))["macro_f1"]
    res["with_bias"], res["with_bias_consistency"] = s0, s1
    choice = {"decode": how, "bias": bias, "consistency": s1 > s0}
    log(f"dev tuning: {json.dumps({k: round(v, 4) for k, v in res.items()})} -> {choice}")
    return choice, res

def error_analysis(truth, pred):
    cats = {l: Counter() for l in LABELS}; confusion = Counter()
    for did in truth:
        g = set(truth[did]); p = set(pred.get(did, []))
        for s, e, l in g - p:
            ov = [(ps, pe, pl) for ps, pe, pl in p if ps < e and s < pe]
            if any(pl == l for *_, pl in ov): cats[l]["FN_boundary"] += 1
            elif any((ps, pe) == (s, e) for ps, pe, _ in ov):
                cats[l]["FN_label"] += 1; confusion[(l, next(pl for ps, pe, pl in ov if (ps, pe) == (s, e)))] += 1
            elif ov: cats[l]["FN_boundary+label"] += 1
            else: cats[l]["FN_missed"] += 1
        for s, e, l in p - g:
            ov = [x for x in g if x[0] < e and s < x[1]]
            cats[l]["FP_spurious" if not ov else "FP_overlap"] += 1
    return {"per_label": {l: dict(c) for l, c in cats.items()},
            "label_confusions(gold->pred)": {f"{a}->{b}": n for (a, b), n in confusion.most_common(20)}}

def write_predictions(pred, docs, path):
    out = {}
    for d in docs:
        lst, seen = [], set()
        for s, e, lab in sorted(pred.get(d["id"], [])):
            e2 = e - END_SHIFT
            assert isinstance(s, int) and isinstance(e2, int) and 0 <= s < e2 and lab in LABELS
            if (s, e2, lab) not in seen: seen.add((s, e2, lab)); lst.append([s, e2, lab])
        out[d["id"]] = lst
    assert set(out) == {d["id"] for d in docs}
    save_json(out, path); return out

def make_zip(pred_json_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_json_path, "prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
        obj = json.loads(z.read("prediction.json"))
    assert set(obj) == {d["id"] for d in TEST_DOCS}, "submission ids != test ids"

## 10. Training loop and experiment runner
AdamW (weight decay on matrices only), linear warmup then linear decay, fp16 autocast + gradient scaling on GPU,
gradient clipping. The backbone and the new head get separate learning rates (a fresh head needs a much larger one).
After every epoch the dev set is decoded; the best epoch's trainable weights are kept.

In [12]:
def load_teacher(name, device):
    tdir = EXPS_DIR / name
    tcfg = dict(DEFAULTS); tcfg.update(load_json(tdir / "config.json") or load_json(find_artifact(f"p2/exps/{name}/config.json")) or {})
    wpath = find_artifact(f"p2/exps/{name}/trainable_devmodel.pt")
    assert wpath is not None, f"teacher weights p2/exps/{name}/trainable_devmodel.pt not found — attach the run that trained it"
    assert tcfg["head"] == "linear", "the teacher must have a linear (softmax) head"
    teacher, _ = build_ner_model(tcfg, device)
    sd = torch.load(wpath, map_location="cpu", weights_only=False)
    missing = teacher.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return teacher, tcfg

def teacher_targets(name, docs, device, log):
    """teacher's per-token log-probabilities for every document (merged over its own windows) — independent of the
    student's window/header settings, so the student just looks up its tokens"""
    teacher, tcfg = load_teacher(name, device)
    t0 = time.time(); out = doc_emissions(teacher, docs, tcfg, device)
    tot, _ = count_parameters(teacher)
    log(f"teacher {name} ({tcfg['model']}, {tot/1e6:.1f}M params) soft targets for {len(docs)} docs in {time.time()-t0:.0f}s")
    del teacher
    if device.type == "cuda": torch.cuda.empty_cache()
    return out, tot

def soft_batch(chunk, soft, Lc, device):
    out = np.zeros((len(chunk), Lc, NT), np.float32)
    for b, e in enumerate(chunk): out[b, :e["len"]] = soft[e["doc"]][e["start"]:e["start"] + e["len"]]
    return torch.from_numpy(out).to(device)

def train_model(cfg, train_docs, device, epochs, log, dev_docs=None):
    soft, teacher_params = (None, None)
    if cfg["teacher"]:
        assert cfg["head"] == "linear", "KD student uses a linear head"
        soft, teacher_params = teacher_targets(cfg["teacher"], train_docs, device, log)
    model, pcfg = build_ner_model(cfg, device)
    total, trainable = count_parameters(model)
    rng = np.random.default_rng(cfg["seed"])
    head_params = [p for p in model.head.parameters() if p.requires_grad]
    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
    groups = [{"params": [p for p in head_params if p.ndim >= 2], "lr": cfg["lr_head"], "weight_decay": cfg["weight_decay"]},
              {"params": [p for p in head_params if p.ndim < 2], "lr": cfg["lr_head"], "weight_decay": 0.0}]
    if bb_params and cfg["lr"] > 0:
        groups += [{"params": [p for p in bb_params if p.ndim >= 2], "lr": cfg["lr"], "weight_decay": cfg["weight_decay"]},
                   {"params": [p for p in bb_params if p.ndim < 2], "lr": cfg["lr"], "weight_decay": 0.0}]
    else:
        for p in bb_params: p.requires_grad = False
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    for g in opt.param_groups: g["base_lr"] = g["lr"]
    probe = [ex for d in train_docs for ex in windows_for(d, cfg, True, None)]
    n_win = len(probe); n_rare = sum(bool(RARE_TAGS & set(ex["tags"].tolist())) for ex in probe)
    steps_per_epoch = math.ceil((n_win + n_rare * cfg["oversample"]) / cfg["batch"])
    total_steps = max(1, epochs * steps_per_epoch); warm = max(1, int(cfg["warmup"] * total_steps))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = class_weights(cfg["class_weight"]); cw = cw.to(device) if cw is not None else None
    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    tset = set(trainable_names)
    log(f"model {cfg['model']} ({pcfg['arch']}) attn={cfg['attn']} method={cfg['method']} head={cfg['head']}"
        f"{'+bilstm' if cfg['lstm_hidden'] else ''} | params {total/1e6:.1f}M, trainable {trainable/1e6:.3f}M "
        f"({100*trainable/total:.2f}%) | {n_win} windows/epoch, {total_steps} steps")
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    hist, best = [], (-1.0, None, 0)
    step, t0 = 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        exs = [ex for d in train_docs for ex in windows_for(d, cfg, True, rng)]
        if cfg["oversample"]:
            exs += [ex for ex in exs if RARE_TAGS & set(ex["tags"].tolist())] * cfg["oversample"]
        order = rng.permutation(len(exs)); run_loss, nb = 0.0, 0
        for i in range(0, len(order), cfg["batch"]):
            chunk = [exs[j] for j in order[i:i + cfg["batch"]]]
            idx, valid, offs, tags, cmask = collate(chunk, device)
            lr_mult = (step + 1) / warm if step < warm else max(0.0, (total_steps - step) / max(1, total_steps - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_mult
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                em = model(idx, valid)
            em = content(em.float(), offs, tags.size(1))
            if model.head.crf is not None:
                loss = model.head.crf.nll(em, tags, cmask)
            else:
                loss = F.cross_entropy(em.reshape(-1, NT), tags.reshape(-1), weight=cw, ignore_index=-100)
            if soft is not None:                                   # knowledge distillation (Hinton et al. 2015)
                T, a = cfg["kd_T"], cfg["kd_alpha"]
                t = soft_batch(chunk, soft, tags.size(1), device) / T
                kl = (F.softmax(t, -1) * (F.log_softmax(t, -1) - F.log_softmax(em / T, -1))).sum(-1)
                kl = (kl * cmask).sum() / cmask.sum().clamp(min=1)
                loss = a * loss + (1 - a) * T * T * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg["grad_clip"])
            scaler.step(opt); scaler.update()
            run_loss += loss.item(); nb += 1; step += 1
        msg = {"epoch": ep, "loss": run_loss / max(nb, 1), "min": (time.time() - t0) / 60}
        if dev_docs is not None:
            ems = doc_emissions(model, dev_docs, cfg, device)
            f1 = strict_scores(TRUTH_DEV, decode(ems, dev_docs, "viterbi", crf_numpy(model)))["macro_f1"]   # constrained Viterbi
            msg["dev_f1"] = f1
            if f1 > best[0]:
                best = (f1, {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if n in tset}, ep)
        hist.append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
    info = {"total_params": total, "trainable_params": trainable, "train_min": (time.time() - t0) / 60,
            "peak_mem_gb": torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None, "history": hist,
            "windows_per_epoch": n_win, "backbone_arch": pcfg["arch"], "teacher_params": teacher_params}
    if dev_docs is not None and best[1] is not None:
        model.load_state_dict(best[1], strict=False)
        info.update(best_epoch=best[2], best_dev_f1_viterbi=best[0])
    return model, info, trainable_names

def save_trainable(model, names, path):
    sd = {n: p.detach().half().cpu() if p.is_floating_point() else p.detach().cpu() for n, p in model.named_parameters() if n in set(names)}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return load_json(d / "result.json")
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    # 1) train on the train split, select the epoch on dev
    model, info, names = train_model(cfg, TRAIN_DOCS, device, cfg["epochs"], log, dev_docs=DEV_DOCS)
    t0 = time.time(); ems_dev = doc_emissions(model, DEV_DOCS, cfg, device)
    info["infer_sec_per_doc"] = (time.time() - t0) / max(1, len(DEV_DOCS))
    crfp = crf_numpy(model)
    choice, tuning = tune_on_dev(ems_dev, cfg, crfp, log)
    pred_dev = decode(ems_dev, DEV_DOCS, choice["decode"], crfp, choice["bias"], choice["consistency"])
    sc_dev = strict_scores(TRUTH_DEV, pred_dev)
    save_json({"scores": sc_dev, "tuning": tuning, "choice": choice}, d / "dev_metrics.json")
    save_json(error_analysis(TRUTH_DEV, pred_dev), d / "dev_error_analysis.json")
    write_predictions(pred_dev, DEV_DOCS, d / "dev_prediction.json")
    save_trainable(model, names, d / "trainable_devmodel.pt")
    # 2) test predictions with the dev-selected model
    ems_test = doc_emissions(model, TEST_DOCS, cfg, device)
    write_predictions(decode(ems_test, TEST_DOCS, choice["decode"], crfp, choice["bias"], choice["consistency"]),
                      TEST_DOCS, d / "prediction_devmodel.json")
    final_src = "prediction_devmodel.json"
    # 3) optional: retrain on all 70 documents for the selected number of epochs
    if cfg["final"] == "retrain_all":
        del model; torch.cuda.empty_cache() if device.type == "cuda" else None
        log(f"retraining on train+dev for {info.get('best_epoch', cfg['epochs'])} epochs")
        model, _, names = train_model(cfg, TRAIN_DOCS + DEV_DOCS, device, info.get("best_epoch", cfg["epochs"]), log)
        save_trainable(model, names, d / "trainable_all.pt")
        crfp = crf_numpy(model); ems_test = doc_emissions(model, TEST_DOCS, cfg, device)
        write_predictions(decode(ems_test, TEST_DOCS, choice["decode"], crfp, choice["bias"], choice["consistency"]),
                          TEST_DOCS, d / "prediction_all.json")
        final_src = "prediction_all.json"
    make_zip(d / final_src, d / "submission.zip")
    row = {"name": name, "model": cfg["model"], "arch": info["backbone_arch"], "attn": cfg["attn"] if cfg["model"] != "B" else "bidir(enc)",
           "init": cfg["init"], "method": cfg["method"] + (f"-{cfg['lorma_variant']}" if cfg["method"] == "lorma" else "")
                     + (f"+KD({cfg['teacher']})" if cfg["teacher"] else ""),
           "head": cfg["head"] + ("+bilstm" if cfg["lstm_hidden"] else ""), "win": cfg["win_len"], "header": cfg["header_len"],
           "class_w": cfg["class_weight"], "oversample": cfg["oversample"],
           "dev_macro_f1": sc_dev["macro_f1"], "dev_macro_p": sc_dev["macro_p"], "dev_macro_r": sc_dev["macro_r"],
           "dev_f1_raw": max(v for k, v in tuning.items() if k.startswith("raw_")),
           **{f"F1_{l}": sc_dev["per_label"][l]["F1"] for l in LABELS},
           "best_epoch": info.get("best_epoch"), "epochs": cfg["epochs"],
           "total_params_M": info["total_params"] / 1e6,
           "teacher_params_M": (info["teacher_params"] or 0) / 1e6, "trainable_params_M": info["trainable_params"] / 1e6,
           "trainable_%": 100 * info["trainable_params"] / info["total_params"],
           "peak_mem_GB": info["peak_mem_gb"], "train_min": info["train_min"], "infer_s_per_doc": info["infer_sec_per_doc"],
           "decode": choice["decode"], "bias_tuned": bool(choice["bias"]), "consistency": choice["consistency"],
           "submission": f"p2/exps/{name}/submission.zip ({final_src})"}
    save_json({"row": row, "info": info, "cfg": cfg}, d / "result.json")
    log(f"DONE dev macro-F1 {sc_dev['macro_f1']:.4f} (P {sc_dev['macro_p']:.4f} R {sc_dev['macro_r']:.4f})")
    logf.close(); del model
    if device.type == "cuda": torch.cuda.empty_cache()
    return row

## 11. Unit tests (CPU, a few seconds)
1. CRF loss = brute-force `log Σ exp(score)` over all tag sequences; Viterbi = brute-force best sequence.
2. LoRA, LoRMA⁺, LoRMA-π, BitFit, adapters reproduce the frozen layer exactly at initialisation.
3. Windows (with header + overlap) + decoding of *perfect* (one-hot gold) scores reproduce the gold spans exactly
   up to the P2_01 alignment ceiling.

In [13]:
import itertools
def _tests():
    torch.manual_seed(0); n, T = 3, 4
    allowed, start_ok = np.ones((n, n), bool), np.ones(n, bool)
    crf = CRF(n, allowed, start_ok)
    with torch.no_grad():
        crf.trans.normal_(); crf.start.normal_(); crf.end.normal_()
    em = torch.randn(1, T, n); tags = torch.tensor([[0, 2, 1, 1]]); mask = torch.ones(1, T, dtype=torch.bool)
    def sc(y):
        s = crf.start[y[0]] + em[0, 0, y[0]] + crf.end[y[-1]]
        for t in range(1, T): s = s + crf.trans[y[t - 1], y[t]] + em[0, t, y[t]]
        return s
    allp = list(itertools.product(range(n), repeat=T))
    logZ = torch.logsumexp(torch.stack([sc(y) for y in allp]), 0)
    assert abs(T * crf.nll(em, tags, mask).item() - (logZ - sc(tags[0].tolist())).item()) < 1e-4, "CRF forward algorithm"   # nll is per token
    best = max(allp, key=lambda y: sc(y).item())
    tr, st, en = (x.detach().numpy() for x in crf.full())
    assert tuple(viterbi_batch([em[0].numpy()], tr, st, en)[0].tolist()) == best, "Viterbi"
    # masked (shorter) sequence: same as running on the prefix
    em2 = torch.cat([em, torch.randn(1, 2, n)], 1); tags2 = torch.cat([tags, torch.zeros(1, 2, dtype=torch.long)], 1)
    mask2 = torch.tensor([[True] * T + [False] * 2])
    assert abs(crf.nll(em2, tags2, mask2).item() - crf.nll(em, tags, mask).item()) < 1e-4, "CRF masking"
    # adapters start as the identity
    base = nn.Linear(16, 24, bias=False); x = torch.randn(5, 16); y = base(x)
    for mod in (LoRALinear(base, 4, 8), LoRMALinear(base, 4, 8, "plus"), LoRMALinear(base, 4, 8, "pi"),
                BiasLinear(base), AdapterLinear(base, 6)):
        assert torch.allclose(mod(x), y, atol=1e-5), type(mod).__name__
    # windows + merge + decode with perfect scores reproduce gold (up to the alignment ceiling)
    cfg = dict(DEFAULTS, win_len=256 if P2_SMOKE else 512, header_len=64)
    docs = [d for d in TRAIN_DOCS if len(d["ids"])][:4]
    oracle = {}
    for d in docs:
        acc = np.zeros((len(d["ids"]), NT), np.float32); ws = np.zeros(len(d["ids"]), np.float32)
        for ex in windows_for(d, cfg, train=False):
            j = np.arange(ex["len"]); w = np.minimum(j + 1, ex["len"] - j)
            acc[ex["start"]:ex["start"] + ex["len"]] += np.eye(NT, dtype=np.float32)[ex["tags"]] * w[:, None]
            ws[ex["start"]:ex["start"] + ex["len"]] += w
        assert (ws > 0).all(), "a token not covered by any window"
        oracle[d["id"]] = np.log(acc / ws[:, None] + 1e-9)
    direct = {d["id"]: doc_spans_from_tags(d, d["tags"]) for d in docs}
    for how in ("greedy", "viterbi"):
        assert decode(oracle, docs, how) == direct, f"window merge/decode ({how})"
    truth = {d["id"]: [tuple(x) for x in d["spans"]] for d in docs}
    print(f"unit tests passed | oracle strict macro-F1 on 4 train docs = {strict_scores(truth, direct)['macro_f1']:.4f}")
_tests()

unit tests passed | oracle strict macro-F1 on 4 train docs = 1.0000


## 12. Run the queue
One worker thread per GPU pulls experiment names from the queue (CUDA kernels run outside Python's lock, so two
threads keep two GPUs busy). Each experiment is independent and saved as soon as it finishes; a stopped commit
can be resumed by attaching its output — finished experiments are skipped.

In [14]:
def run_queue(names):
    q = _queue.Queue()
    for n in names: q.put(n)
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            try: name = q.get_nowait()
            except _queue.Empty: return
            if time_left_h() < MIN_HOURS_PER_EXP:
                print(f"[{name}] skipped: only {time_left_h():.2f} h left in this session"); continue
            try:
                if dev.type == "cuda":
                    with torch.cuda.device(dev): run_experiment(name, dev)
                else:
                    run_experiment(name, dev)
            except Exception:
                print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                (EXPS_DIR / name).mkdir(parents=True, exist_ok=True)
                (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

t_queue = time.time()
run_queue(RUN_QUEUE)
print(f"queue finished in {(time.time() - t_queue)/60:.1f} min")

[f2_E_lorma | cuda:0] model E (prefixlm) attn=bidir method=lorma head=linear | params 113.2M, trainable 3.067M (2.71%) | 1454 windows/epoch, 6050 steps[f3_F_causal_adapter | cuda:1] model F (efficient) attn=causal method=adapter head=linear+bilstm | params 112.3M, trainable 4.493M (4.00%) | 1454 windows/epoch, 6050 steps

[f3_F_causal_adapter | cuda:1] {"epoch": 1, "loss": 1.0662, "min": 0.9323, "dev_f1": 0.3738}
[f2_E_lorma | cuda:0] {"epoch": 1, "loss": 1.0707, "min": 1.1516, "dev_f1": 0.3603}
[f3_F_causal_adapter | cuda:1] {"epoch": 2, "loss": 0.1972, "min": 2.0799, "dev_f1": 0.4847}
[f2_E_lorma | cuda:0] {"epoch": 2, "loss": 0.2256, "min": 2.5708, "dev_f1": 0.4879}
[f3_F_causal_adapter | cuda:1] {"epoch": 3, "loss": 0.116, "min": 3.2241, "dev_f1": 0.5248}
[f2_E_lorma | cuda:0] {"epoch": 3, "loss": 0.148, "min": 3.9952, "dev_f1": 0.4928}
[f3_F_causal_adapter | cuda:1] {"epoch": 4, "loss": 0.0755, "min": 4.3694, "dev_f1": 0.5324}
[f2_E_lorma | cuda:0] {"epoch": 4, "loss": 0.1066, "mi

## 13. Results (all experiments found here or in attached earlier runs)
`dev_macro_f1` = strict macro-F1 on our 14 dev documents after the dev-tuned decoding; `dev_f1_raw` = before any
tuning. Each row's `submission.zip` is ready to upload (`p2/exps/<name>/submission.zip`).

In [15]:
rows = []
for f in sorted(EXPS_DIR.glob("*/result.json")):
    rows.append(load_json(f)["row"])
if rows:
    df = pd.DataFrame(rows).sort_values("dev_macro_f1", ascending=False)
    df.to_csv(ROOT / "p2" / "p2_experiments.csv", index=False)
    cols = ["name", "model", "attn", "init", "method", "head", "win", "header", "dev_macro_f1", "dev_f1_raw",
            "dev_macro_p", "dev_macro_r", "trainable_params_M", "trainable_%", "peak_mem_GB", "train_min", "best_epoch"]
    display(df[cols].round(4).to_string(index=False))
    print("\nper-label dev F1:")
    display(df[["name"] + [f"F1_{l}" for l in LABELS]].round(3).to_string(index=False))
    best_name = df.iloc[0]["name"]
    print(f"\nbest so far: {best_name} -> {ROOT}/p2/exps/{best_name}/submission.zip")
    ea = load_json(EXPS_DIR / best_name / "dev_error_analysis.json")
    print("error analysis of the best experiment:")
    display(pd.DataFrame(ea["per_label"]).T.fillna(0).astype(int).to_string())
    print("most frequent label confusions (gold -> predicted):", ea["label_confusions(gold->pred)"])
for f in sorted(EXPS_DIR.glob("*/error.txt")):
    print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

'               name model   attn       init     method          head  win  header  dev_macro_f1  dev_f1_raw  dev_macro_p  dev_macro_r  trainable_params_M  trainable_%  peak_mem_GB  train_min  best_epoch\nf3_F_causal_adapter     F causal pretrained    adapter linear+bilstm  512     256        0.6029      0.5783       0.5898       0.6513              4.4933       4.0028       2.4369    28.6492           9\n         f2_E_lorma     E  bidir pretrained lorma-plus        linear  512     256        0.5853      0.5608       0.5853       0.5926              3.0666       2.7094       2.9324    35.3595          18'


per-label dev F1:


'               name  F1_APP  F1_RESP  F1_A.COUNSEL  F1_R.COUNSEL  F1_JUDGE  F1_WIT  F1_AUTH  F1_COURT  F1_STAT  F1_PREC  F1_DATE  F1_CASENO\nf3_F_causal_adapter   0.264    0.480         0.775         0.781     0.779   0.259    0.281     0.751    0.850    0.474    0.929      0.610\n         f2_E_lorma   0.194    0.325         0.684         0.754     0.747   0.200    0.451     0.787    0.855    0.465    0.927      0.637'


best so far: f3_F_causal_adapter -> /kaggle/working/cs781_p1/p2/exps/f3_F_causal_adapter/submission.zip
error analysis of the best experiment:


'           FN_missed  FN_boundary+label  FN_label  FN_boundary  FP_overlap  FP_spurious\nAPP               45                  8        10            3          19            4\nRESP               6                  8         2            5          34           49\nA.COUNSEL          0                  1         5            0           8            4\nR.COUNSEL          1                  0         6            0           6            1\nJUDGE              2                  0         2            4           5            8\nWIT                6                  5         7            3          27           72\nAUTH             138                  5         4           11          11           51\nCOURT             12                  4         1           29          52           17\nSTAT              15                  0         0           12          15           50\nPREC              30                  3        47           75          83           73\nDATE               3

most frequent label confusions (gold -> predicted): {'PREC->COURT': 22, 'PREC->RESP': 11, 'PREC->WIT': 9, 'WIT->RESP': 7, 'APP->WIT': 6, 'R.COUNSEL->A.COUNSEL': 6, 'A.COUNSEL->R.COUNSEL': 5, 'APP->RESP': 3, 'PREC->JUDGE': 2, 'JUDGE->PREC': 2, 'AUTH->PREC': 2, 'AUTH->RESP': 2, 'PREC->APP': 2, 'COURT->PREC': 1, 'DATE->CASENO': 1, 'PREC->A.COUNSEL': 1, 'RESP->APP': 1, 'APP->A.COUNSEL': 1, 'RESP->R.COUNSEL': 1}
